#  APIs 

## From Databases to APIs

In the previous notebook, we worked with data already stored in a database — structured, reliable, and ready for analysis.

But not all data lives neatly inside our systems. 

In many analytics roles, valuable data comes from external sources — like weather, marketing platforms, social media, or financial services.

To access that kind of live or third-party data, analysts use APIs (Application Programming Interfaces).

APIs let us connect directly to web-based data sources, retrieve information (usually in JSON format), and integrate it into our workflows.

This lesson will show you how to extract data from an API, inspect it, and prepare it for use in analytics — just like we did with SQL and Pandas, but this time from the web.


This notebook will show how to:

1. Understand JSON, the language of APIs.

2. Use Python’s requests library to access API endpoints.

3. Convert API responses into Pandas DataFrames.

4. Prepare for integrating this data into your analytics pipeline.

Data Pipeline Context

This notebook extends your previous learning and fits into the broader analytics flow:

```zsh
API (Extract) → Database (Load) → Pandas (Transform) → Visualization (Insights)
```

--------

## 1. Warm-Up: JSON — The Language of APIs

Before we work with APIs, let’s recall how JSON looks and works.


`JSON (JavaScript Object Notation)` is a text-based format for representing structured data — like dictionaries and lists in Python.



In [ ]:
import json

In [ ]:

# Creating student records
student1 = {
    "name": "Alice",
    "age": 15,
    "grade": 10,
    "subjects": ["Math", "Science", "History"]
}

student2 = {
    "name": "Bob",
    "age": 16,
    "grade": 11,
    "subjects": ["English", "Physics", "Geography"]
}

student3 = {
    "name": "Carol",
    "age": 14,
    "grade": 9,
    "subjects": ["Art", "Music", "PE"]
}

# Creating a list of student records
student_database = [student1, student2, student3]

# Converting the student database to JSON format
json_data = json.dumps(student_database, indent=4)

# Printing the JSON data
print(json_data)

# Checking the type
type(json_data)

- What kind of structure do you see?

- How do the key-value pairs resemble Python dictionaries?

- Why might this format be useful for transferring data between systems?

You can convert JSON strings back into Python objects with `json.loads()`:

In [ ]:
parsed_data = json.loads(json_data)


In [ ]:
type(parsed_data)

In [ ]:
parsed_data

## 2. What is an API?

An API is a messenger between two pieces of software — it defines how they communicate and exchange data.

When we talk about Web APIs, we mean services available over the internet that allow programs to:

- Request data from a server

- Receive it in a structured, machine-readable format (often JSON)

For data analysts, APIs are essential because they let us pull live or external data into our analysis workflows.

###  REST APIs and HTTP

Most web APIs follow REST (Representational State Transfer) principles and communicate using HTTP, the same protocol your browser uses.

Common HTTP methods you’ll encounter:

| Method | Purpose              | Analogy                     |
| ------ | -------------------- | --------------------------- |
| GET    | Retrieve data        | “Show me this information.” |
| POST   | Create new data      | “Add this record.”          |
| PUT    | Update existing data | “Edit this record.”         |
| DELETE | Remove data          | “Delete this record.”       |


<img src='webapi_waiter.png' style="background-color:white;">
> https://medium.com/@ama.thanu/what-is-an-api-how-does-it-work-f4ea552d741f

------------------------------------------------------

## 3. Accessing Data with Python’s `requests` Library

The Python `requests` library is a popular third-party library that simplifies the process of making HTTP requests and working with HTTP responses. It provides a high-level interface for sending HTTP requests to web servers and receiving their responses. This library is widely used for tasks like fetching data from APIs, sending data to servers, and interacting with web resources.
> #### Intro to request library:
>- https://requests.readthedocs.io (Documentation)
>- https://realpython.com/python-requests/
>- https://www.geeksforgeeks.org/response-json-python-requests/

### Installation

Make sure that nf_sql env is active in the terminal as well as in the jupyter notebook kernel.

If not:
`conda activate nf_sql`

Copy and run the following command in the terminal

`conda install conda-forge::requests`


In [ ]:
import requests
import pprint # https://docs.python.org/3/library/pprint.html
import pandas as pd

### Example: Getting Data from the Berlin Public Transport API

The VBB (Verkehrsverbund Berlin-Brandenburg) API provides public transport data.
Let’s start by searching for “Moritzplatz”:

In [ ]:
api_url = 'https://v6.vbb.transport.rest/locations?query=moritzplatz'

[v6.vbb.transport.rest](https://v6.vbb.transport.rest) is a free to use REST API for the public transportation system of Berlin & Brandenburg, VBB. The API doesn't require a sign up or a token.


All Transit APIs can be found her https://transport.rest/

In [ ]:
response = requests.request("GET", api_url)

# If you see 200, the request succeeded ✅.
print(response.status_code)

### Inspecting the Response

The response contains text (a JSON string) that we can inspect in different ways:
**All status codes can be found here:** https://http.cat/

The response objects allow you to access the contents of the response. Try them out!

In [ ]:
# response.content # we can read the contents in raw form
print(response.text[:500]) # we can read the contents as text limit to 500 chars.
#response.json() # we can translate response to json
# response.headers
# response.url


In [ ]:
# let's  translate our original response to json

result = response.json()

In [ ]:
result

# check also out the list items: result[0] and so on

In [ ]:
pprint.pprint(result[:2])  # Print only the first 2 entries nicely

> #### SIDEBAR:
> online tools like https://jsonformatter.org/ can also help to review JSON 

## 4. Working with API Data in Pandas

Let’s transform that response into a Pandas DataFrame.

This helps us explore, clean, and visualize data using our familiar toolkit.

In [ ]:
# Convert directly into a DataFrame
df = pd.DataFrame(result)
df.head()

### Inspect the Structure

Take note of:

- Which columns contain text, lists, or nested structures?

- How might you flatten or expand this data to make it easier to analyze?

### Alternative: Reading JSON Directly with Pandas

Pandas can read JSON directly from a URL too:

In [ ]:
pd.read_json(api_url).head()

## 5. Saving and Loading JSON Data

You can save the JSON response for later use — useful for caching or offline analysis.

In [ ]:
with open('vbb_example.json', 'w') as f:
    json.dump(result, f, indent=4)

# Load it again
with open('vbb_example.json', 'r') as f:
    data_loaded = json.load(f)

print(len(data_loaded))


## 6. Evaluating APIs

When choosing an API for analytics, consider:

| Question                                  | Why It Matters                       |
| ----------------------------------------- | ------------------------------------ |
| How much data can I retrieve per request? | Some APIs have call limits.          |
| Does it require authentication?           | Many APIs use tokens or keys.        |
| Is it free or paid?                       | Check pricing tiers and rate limits. |
| Is documentation clear?                   | Good examples make API use easier.   |
| What data format is returned?             | JSON is easiest to parse in Python.  |


## 7. Food for Thought

You now know how to extract data from external sources using APIs.
In the next module, you’ll learn how to integrate this data into your database, transform it for analysis, and visualize insights.

By now, you’ve touched all the components of a real analytics workflow:

`API (Extract) → Database (Load) → Pandas (Transform) → Visualization (Insight)`


Welcome to the real data pipeline of modern analytics. 🚀

> Next: Challenge: Your First API Exploration


---------

# Additional Reading Recommended!

### Are APIs free?

there are
- Free APIs (Meteostat API https://dev.meteostat.net/api/) 
- Paid APIs (Bloomberg API https://www.bloomberg.com/professional/)
- Hybrid APIs (OpenWeatherAPI https://openweathermap.org/api)

### What to consider when searching for an API?

- How much data is available?
- How much data can be retrieved in one call
- Is there a call rate limit? (e.g. you are not allowed to make more than X requests per hour)
- Does the API require authentication?
    - Is the authentication difficult (username/password: easy; OAuth (more complex; need to create an account, generate authentication tokens)
- Is there a fee? (e.g. you are charged X cents per request)
- Is the documentation easy to understand? Usually a few example URLs are super valuable in trying to understand the syntax.
- What type of data is returned? JSON is usually your best bet.

### What makes a web API better than scraping? 

- Interface is not meant for humans but for machines. This refers both, to  
    - requests (=> well defined way how to ask for information)
    - and to data delivery (=> provides data in a **more machine-readable format**, mostly json) 
- For larger data providers (e.g. reddit) there are often  
    - helper libraries  
    - documentation and examples
    - clear terms and conditions about what is allowed
- From the point of view of the provider, APIs
    - create visibility (e.g. for a hotel operator whose vacant rooms are to be found by a meta search engine)
    - ...

### How to *find* an interesting (web) API?

(Please have a look (!) and feel free to use your desired one for this week's project.)  
- just search the internet  
- have a look [here](https://apislist.com/). Tons of APIs!
- or [here](https://dev.to/biplov/15-fun-apis-for-your-next-project-5053) Fun APIs!
- But usually you want to start the other way round:  
 You know which data you want to scrape and specifically look for whether an API exists for this [data](https://spiced.space/bergamot-encoder/ds-course/chapters/final_project/datasets.html#datasets).  

-------------------------------------

## API Acess Tokens - how do they work?

In [ ]:
from IPython.display import IFrame

In [ ]:
IFrame("https://www.youtube.com/embed/DanUVSlOSQQ", width=560, height=315)


<iframe width="560" height="315" src="https://www.youtube.com/embed/DanUVSlOSQQ" frameborder="0" allowfullscreen></iframe>


<iframe width="560" height="315" src="https://www.youtube.com/embed/DanUVSlOSQQ" frameborder="0" allowfullscreen></iframe>


## Next Step: 

###  Signup and explore meteostat API exercise




------

## Additional Reading Optional

### REST (Representational State Transfer) and RESTful API
### What is REST?


REST (Representational State Transfer) is a way to design web services that allows different systems to communicate over the internet.
It's a set of guidelines that can be used with any protocol (like HTTP), but it's most commonly used with HTTP because that's how most web traffic works.
https://learn.microsoft.com/en-us/azure/architecture/best-practices/api-design#what-is-rest


**RESTful API** is an application program interface (API) that uses [HTTP requests methods](https://www.restapitutorial.com/lessons/httpmethods.htm) GET, PUT, POST and DELETE data. 

Every application which has CRUD(Create, Read, Update, Delete) operation has an API to create data, to get data, to update data or to delete data from database.
  
In summary, REST API is a specific type of Web API that adheres to the principles of Representational State Transfer, using HTTP methods and statelessness. Web API is a more general term that encompasses APIs exposed over the web, which may or may not follow REST principles and can use various protocols and data formats.


### HTTP (Hypertext Transfer Protocol)

HTTP is an established communication protocol between a client and a server. A client in this case is a browser and server is the place where you access data. HTTP is a network protocol used to deliver resources which could be files on the World Wide Web, whether they're HTML files, image files, query results, scripts, or other file types.

A browser is an HTTP client because it sends requests to an HTTP server (Web server), which then sends responses back to the client.

Web API is a set of stpecifications 
- Hypertext Transfer Protocol (HTTP) request messages, 
- along with a definition of the structure of response messages, usually in an XML or a JavaScript Object Notation (JSON) format. 

### OpenAPI 

OpenAPI is a specification used for creating interfaces used in describing, producing, consuming, and visualizing RESTful APIs. It is also known as the swagger specification.

Very informative about what OpenAPI is: https://document360.com/blog/open-api